# Spinful Kagome Kane-Mele TUfRG Pipeline Demo

This notebook demonstrates the four-file workflow:

1. `TUfRG_model.py` — model construction and sanity checks.
2. `TUfRG_runner.py` — divERGe TUfRG flow and postprocess.
3. `TUfRG_diagnosis.py` — q-resolved general diagnosis.
4. `TUfRG_realspace.py` — selected-mode real-space reconstruction.

The notebook uses tiny smoke-test settings. These are for checking the pipeline, not for physics conclusions.

In [ ]:
# Cell 0: check files and versions
from pathlib import Path

for fname in ["TUfRG_model.py", "TUfRG_runner.py", "TUfRG_diagnosis.py", "TUfRG_realspace.py"]:
    p = Path(fname)
    print(f"{fname:22s} exists={p.exists()} size={p.stat().st_size if p.exists() else 0}")
    if p.exists():
        text = p.read_text(errors="replace")
        for marker in ["v2-jsonsafe-2026-05-12", "v2-sector-names-2026-05-12"]:
            if marker in text:
                print("  marker:", marker)

In [ ]:
# Cell 1: import model file and run pure-python sanity checks
import importlib, inspect
import TUfRG_model as km
importlib.reload(km)

print("KMParameters:", inspect.signature(km.KMParameters))
print("MeshParameters:", inspect.signature(km.MeshParameters))

params = km.KMParameters(
    t=1.0,
    U=0.2,
    V=0.0,
    lambda_nn=0.05,
    lambda_nnn=0.0,
    filling_mode="fixed_mu",
    mu=0.0,
    target_filling=5.0/12.0,
    filling_ngrid=40,
)
mesh = km.MeshParameters(nk=4, nkf=1, tu_distance=1.01)

resolved, filling_report = km.resolve_parameters(params)
print("resolved:", resolved)
print("filling_report:", filling_report)

report = km.noninteracting_sanity_report(
    t=resolved.t,
    mu=resolved.mu,
    lambda_nn=resolved.lambda_nn,
    lambda_nnn=resolved.lambda_nnn,
    ngrid=40,
)

for k in [
    "max_hermiticity_error",
    "max_time_reversal_error",
    "max_Sz_commutator_error",
    "max_spin_offdiag_block_norm",
    "max_Hdn_minus_Hup_star_minus_k",
    "estimated_filling_at_fermi_zero",
]:
    print(k, "=", report[k])

km.assert_sanity_passes(report, tol=1e-8)
print("Model sanity checks passed.")

## Run a tiny TUfRG flow

This creates `km_spinful_runs/sanity_runner_nnkm_fixed_mu/` containing:

```text
T.dvg
F.dat
O.dvg
metadata.json
```

In [ ]:
# Cell 2: run a tiny NN Kane-Mele fixed-mu case
!python TUfRG_runner.py \
  --U 0.2 \
  --V 0.0 \
  --lambda-nn 0.05 \
  --lambda-nnn 0.0 \
  --filling-mode fixed_mu \
  --mu 0.0 \
  --nk 4 \
  --nkf 1 \
  --tu-distance 1.01 \
  --niter 3 \
  --maxvert 50.0 \
  --tag sanity_runner_nnkm_fixed_mu

In [ ]:
# Cell 3: inspect runner outputs
from pathlib import Path
import json

case_dir = Path("km_spinful_runs/sanity_runner_nnkm_fixed_mu")
for fname in ["T.dvg", "F.dat", "O.dvg", "metadata.json"]:
    p = case_dir / fname
    print(f"{fname:14s} exists={p.exists()} size={p.stat().st_size if p.exists() else 0}")

meta = json.loads((case_dir / "metadata.json").read_text())
print("\nmesh:", meta.get("mesh"))
print("model_summary keys:", meta.get("model_summary", {}).keys())
print("q_indices:", meta["model_summary"]["metadata"].get("q_indices"))
print("filling_report:", meta["model_summary"]["metadata"].get("filling_report"))

## General q-resolved diagnosis

This scans the first few modes of each selected `(channel, q)` sector. It is designed to catch near-degenerate modes within the same sector and to recommend candidates for real-space reconstruction.

In [ ]:
# Cell 4: run general diagnosis
!python TUfRG_diagnosis.py km_spinful_runs/sanity_runner_nnkm_fixed_mu \
  --q Gamma M1 M2 M3 \
  --channels P C D \
  --n-modes 4 \
  --top-components 8 \
  --degeneracy-tol 1e-3

In [ ]:
# Cell 5: view diagnosis tables
import pandas as pd

diag_dir = case_dir / "diagnosis"
q_summary = pd.read_csv(diag_dir / "q_summary.csv")
mode_summary = pd.read_csv(diag_dir / "mode_summary.csv")
deg_summary = pd.read_csv(diag_dir / "degeneracy_summary.csv")

print("q_summary", q_summary.shape)
display(q_summary.head(12))

print("mode_summary", mode_summary.shape)
display(mode_summary.head(12))

print("degeneracy_summary", deg_summary.shape)
display(deg_summary.head(12))

## Real-space reconstruction

Use this only for selected modes. The example below reconstructs a single-Q PH charge-like pattern and then a 3Q pattern.

In [ ]:
# Cell 6: single-Q real-space reconstruction
!python TUfRG_realspace.py km_spinful_runs/sanity_runner_nnkm_fixed_mu \
  --channel D \
  --sector ph-charge \
  --q Gamma \
  --mode 0 \
  --observable density,bond,current \
  --combine singleQ \
  --supercell 2 2 \
  --template-overlap

In [ ]:
# Cell 7: 3Q real-space reconstruction
!python TUfRG_realspace.py km_spinful_runs/sanity_runner_nnkm_fixed_mu \
  --channel D \
  --sector ph-charge \
  --q M1 M2 M3 \
  --mode 0 0 0 \
  --observable density,bond,current \
  --combine 3Q \
  --theta 0 0 0 \
  --supercell 2 2 \
  --template-overlap

In [ ]:
# Cell 8: view newest real-space output tables and image path
from IPython.display import Image, display

root = case_dir / "realspace"
latest = max([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
print("latest realspace output:", latest)

for fname in ["site_pattern.csv", "bond_pattern.csv", "current_pattern.csv", "pairing_pattern.csv", "pattern_summary.csv", "template_overlap.csv"]:
    p = latest / fname
    if p.exists() and p.stat().st_size > 0:
        df = pd.read_csv(p)
        print("\n", fname, df.shape)
        display(df.head(8))

img = latest / "pattern.png"
if img.exists():
    print("\npattern image:", img)
    display(Image(filename=str(img)))

## Notes

- Tiny smoke-test settings often produce near-zero real-space current/bond amplitudes. In that case template overlaps can be numerically misleading.
- For physics, increase `nk`, `nkf`, `niter`, and scan meaningful `U,V,lambda_nn,lambda_nnn` values.
- `TUfRG_diagnosis.py` should be used before `TUfRG_realspace.py`; do not reconstruct all modes blindly.